In [0]:
hchbadjustmentdetail_path=dbutils.widgets.get("hchbadjustmentdetail_path")
adjustments_unpivot_path=dbutils.widgets.get("adjustments_unpivot_path")
hchbadjustments_path=dbutils.widgets.get("hchbadjustments_path")
office_path = dbutils.widgets.get("office_path")
client_path=dbutils.widgets.get("client_path")
date_path=dbutils.widgets.get("date_path")
payerdimension_path=dbutils.widgets.get("payerdimension_path")
adjustmentcode_path=dbutils.widgets.get("adjustmentcode_path")
adjustmenttype_path=dbutils.widgets.get("adjustmenttype_path")
cubeserviceofficetxnweekendingdate_path=dbutils.widgets.get("cubeserviceofficetxnweekendingdate_path")
mart_fact_adjustments_path=dbutils.widgets.get("mart_fact_adjustments_path")

In [0]:
spark.sql(
  f"""
CREATE OR REPLACE TEMPORARY VIEW temp_hchb AS
SELECT 
  *,
  split(clientname, ',')[0] AS LastName,
  LTRIM(RTRIM(split(clientname, ',')[1])) AS FirstName,
  CASE 
    WHEN LENGTH(branchcode) = 3 AND branchcode LIKE '00%' THEN RIGHT(branchcode, 1)
    WHEN LENGTH(branchcode) = 3 AND branchcode RLIKE '0[0-9].*' THEN RIGHT(branchcode, 2)
    ELSE branchcode 
  END AS branch
FROM {hchbadjustmentdetail_path};
  """
)

In [0]:
spark.sql(
    f"""
TRUNCATE TABLE {adjustments_unpivot_path};
"""
)

spark.sql(
    f"""
INSERT INTO {adjustments_unpivot_path} (
  paid,
  clientname,
  mrnumber,
  allmrnumbers,
  postdate,
  entered,
  shiftdate,
  branchcode,
  adjustment_code,
  adjtype,
  dsc_desc,
  episodeid,
  periodnumber,
  id,
  ptid,
  psid,
  payortype,
  payorname,
  branch,
  firstname,
  lastname,
  adjustmenttype,
  adjustmentamount
)
SELECT 
  paid,
  clientname,
  mrnumber,
  allmrnumbers,
  postdate,
  entered,
  shiftdate,
  branchcode,
  adjustment_code,
  adjtype,
  dsc_desc,
  episodeid,
  periodnumber,
  id,
  ptid,
  psid,
  payortype,
  payorname,
  branch,
  firstname,
  lastname,
  adjustmenttype,
  CAST(adjustmentamount AS DECIMAL(15,4)) AS adjustmentamount
FROM (
  SELECT 
    paid, clientname, mrnumber, allmrnumbers, postdate, entered, shiftdate,
    branchcode, adjustment_code, adjtype, dsc_desc, episodeid, periodnumber,
    id, 
    ptid, psid, payortype, payorname,
    branch, firstname, lastname,
    'manadj' AS adjustmenttype, manadj AS adjustmentamount
  FROM temp_hchb
  WHERE manadj <> '0.0000'
  
  UNION ALL
  
  SELECT 
    paid, clientname, mrnumber, allmrnumbers, postdate, entered, shiftdate,
    branchcode, adjustment_code, adjtype, dsc_desc, episodeid, periodnumber,
    id, 
    ptid, psid, payortype, payorname,
    branch, firstname, lastname,
    'sysadj' AS adjustmenttype, sysadj AS adjustmentamount
  FROM temp_hchb
  WHERE sysadj <> '0.0000'
);
"""
)

In [0]:
spark.sql(
    f"""
    TRUNCATE TABLE {hchbadjustments_path};
    """
)

spark.sql(
    f"""
INSERT INTO {hchbadjustments_path} (
    reportingweekendingdatekey,
    sourcesystemkey,
    clientkey,
    firstname,
    lastname,
    posteddatetkey,
    payorkey,
    adjustmenttypekey,
    adjustment_code_key,
    adjustmentamount,
    officekey,
    reportingweekendingdate
)
WITH ReportingWeekCalculation AS (
    SELECT 
        t.*,
        CASE dayofweek(t.postdate)
            WHEN 1 THEN t.postdate
            WHEN 2 THEN date_add(t.postdate, -1)
            WHEN 3 THEN date_add(t.postdate, -2)
            WHEN 4 THEN date_add(t.postdate, -3)
            WHEN 5 THEN date_add(t.postdate, 3)
            WHEN 6 THEN date_add(t.postdate, 2)
            WHEN 7 THEN date_add(t.postdate, 1)
        END AS reportingweekendingdate
    FROM {adjustments_unpivot_path} t
)
SELECT 
    W.WeekEndingDateKey AS reportingweekendingdatekey,
    6 AS sourcesystemkey,
    c.ClientKey AS clientkey,
    c.FirstName AS firstname,
    c.LastName AS lastname,
    d.DateKey AS posteddatetkey,
    P.PayerKey AS payorkey,
    AT.AdjustmentTypeKey AS adjustmenttypekey,
    AC.Adjustment_Code_Key AS adjustment_code_key,
    CAST(t.adjustmentamount AS DOUBLE) AS adjustmentamount,
    O.OfficeKey AS officekey,
    t.reportingweekendingdate
FROM ReportingWeekCalculation t
LEFT JOIN (
    SELECT MAX(ClientKey) AS ClientKey, 
           TRIM(cl.FirstName) AS FirstName, 
           TRIM(cl.LastName) AS LastName, 
           OfficeNumber
    FROM {client_path} cl
    WHERE SourceSystem = 'HCHB'
    GROUP BY TRIM(cl.FirstName), TRIM(cl.LastName), OfficeNumber
) c ON TRIM(LOWER(c.FirstName)) = TRIM(LOWER(t.firstname))
    AND TRIM(LOWER(c.LastName)) = TRIM(LOWER(t.lastname))
    AND c.OfficeNumber = t.branch
LEFT JOIN {office_path} O 
    ON O.OfficeNumber = t.branch
LEFT JOIN {date_path} d 
    ON d.CalendarDate = t.postdate
LEFT JOIN {payerdimension_path} P 
    ON TRIM(P.PayerID) = TRIM(CAST(t.psid AS STRING))
LEFT JOIN {adjustmenttype_path} AT 
    ON TRIM(UPPER(AT.Adjustment_Type)) = TRIM(UPPER(t.adjustmenttype))
LEFT JOIN {adjustmentcode_path} AC 
    ON TRIM(UPPER(AC.Adjustment_Code_Description)) = TRIM(UPPER(t.adjustment_code))
    AND UPPER(AC.Source_System) = 'HCHB'
LEFT JOIN {cubeserviceofficetxnweekendingdate_path} W 
    ON W.WeekEndingDate = t.reportingweekendingdate;
    """
)

In [0]:
spark.sql(
    f"""
    delete from {mart_fact_adjustments_path} where source_system_key=6
    """
)

In [0]:
spark.sql(
    f"""

INSERT INTO {mart_fact_adjustments_path} (
    reporting_week_ending_date_key,
    posted_date_key,
    source_system_key,
    office_key,
    client_key,
    payor_key,
    adjustment_code_key,
    adjustment_type_key,
    invoice_number,
    adjustment_amount
)
SELECT 
    reportingweekendingdatekey AS reporting_week_ending_date_key,
    posteddatetkey AS posted_date_key,
    sourcesystemkey AS source_system_key,
    officekey AS office_key,
    clientkey AS client_key,
    payorkey AS payor_key,
    adjustment_code_key,
    adjustmenttypekey AS adjustment_type_key,
    NULL AS invoice_number,
    CAST(adjustmentamount AS DECIMAL(18,2)) AS adjustment_amount
FROM {hchbadjustments_path};
    """
)